# 04 · Evaluación final y entrega de modelos
Requisitos: 01 y 02 completos. Si ejecutaste 03, se utiliza su selección
por validación. Si lo omitiste, se utiliza YOLO26 elegido de antemano.
**No entrena modelos.** Recupera los registros desde disco, aunque abras
una sesión nueva de Colab. Test requiere confirmar que fijaste los modelos.


## Celda 01 · Conectar Drive y localizar o extraer el proyecto

Repetir en cada cuaderno y después de reiniciar el entorno. Sube `colab_yolo26_listo.zip` a Mi unidad: si falta el proyecto, esta celda lo extrae automáticamente. Conserva los datos existentes y no escribe en `runs/`. Para otras rutas, define `IOTTESIS_ROOT` (carpeta que contiene `scripts/`) o `IOTTESIS_ZIP` (archivo ZIP). Ningún cuaderno depende de las variables de otro.


In [ ]:
# Celda 01 · Conectar Drive y localizar o extraer el proyecto
"""Self-contained Colab startup, also embedded in the notebooks before imports."""
import importlib
from itertools import islice
import os
from pathlib import Path, PurePosixPath
import shutil
import stat
import sys
from zipfile import ZipFile


PROJECT_FILES = (
    'scripts/flujo_colab.py', 'scripts/entrenar_yolo26.py', 'scripts/preparar_datasets_reales.py',
    'requirements-datasets.txt', 'requirements-vision.txt',
)


def iniciar_colab(*, drive_home='/content/drive/MyDrive', content='/content', mount=True):
    """Find a complete project, or install the local bundle without replacing data."""
    drive_home, content = Path(drive_home), Path(content)
    if mount:
        try:
            from google.colab import drive
        except ImportError:
            drive = None
        if drive is not None:
            drive.mount(str(drive_home.parent))

    default_root = drive_home / 'Tesis_IOT_Papa'
    configured = os.environ.get('IOTTESIS_ROOT')
    requested = Path(configured).expanduser().resolve() if configured else None
    # The previous notebook set this default even when the folder did not exist.
    explicit = requested is not None and requested != default_root.resolve()
    bases = [requested] if explicit else [default_root, drive_home / 'IOTtesis',
                                         content / 'IOTtesis', content / 'Tesis_IOT_Papa',
                                         Path.cwd(), *Path.cwd().parents]

    def ready(path):
        return all((path / name).is_file() for name in PROJECT_FILES)

    candidates = list(bases)
    for base in bases:
        candidates.extend(base / name for name in ('Tesis_IOT_Papa', 'IOTtesis'))
        if base.name == 'notebooks':
            candidates.append(base.parent)
    root = next((p.resolve() for p in candidates if ready(p)), None)
    if root is None and not explicit and drive_home.is_dir():
        # Bounded discovery: renamed project folders, and one enclosing folder.
        discovered = set()
        for folder in islice(drive_home.iterdir(), 200):
            if folder.name.startswith('.') or not folder.is_dir():
                continue
            for candidate in (folder, folder / 'Tesis_IOT_Papa', folder / 'IOTtesis'):
                if ready(candidate):
                    discovered.add(candidate.resolve())
        if len(discovered) > 1:
            raise RuntimeError('Hay varios proyectos. Define os.environ["IOTTESIS_ROOT"] con uno de: '
                               + ', '.join(str(p) for p in sorted(discovered)))
        root = next(iter(discovered), None)

    if root is None:
        zip_override = os.environ.get('IOTTESIS_ZIP')
        archives = ([Path(zip_override).expanduser()] if zip_override else
                    [base / 'colab_yolo26_listo.zip' for base in (drive_home, content, Path.cwd())])
        archive = next((p for p in archives if p.is_file()), None)
        if archive is None:
            raise RuntimeError(
                'No se encontró el proyecto completo ni el ZIP.\n'
                'Sube colab_yolo26_listo.zip a Mi unidad y vuelve a ejecutar esta celda.\n'
                f'ZIP esperado: {archives[0]}\n'
                'Si usas otras rutas, define os.environ["IOTTESIS_ROOT"] con la carpeta '
                'que contiene scripts/ y os.environ["IOTTESIS_ZIP"] con el archivo ZIP.\n'
                f'Carpeta de proyecto esperada: {requested or default_root}')
        root = requested if requested is not None else (
            default_root if drive_home.is_dir() else
            (content if content.is_dir() else Path.cwd()) / 'Tesis_IOT_Papa')
        root = root.resolve()
        with ZipFile(archive) as package:
            entries = []
            seen = set()
            for member in package.infolist():
                path = PurePosixPath(member.filename)
                parts = path.parts
                if ('\\' in member.filename or path.is_absolute() or '..' in parts
                        or not parts or parts[0] != 'Tesis_IOT_Papa'
                        or stat.S_ISLNK(member.external_attr >> 16)):
                    raise ValueError('Ruta inválida en el ZIP: ' + member.filename)
                if len(parts) == 1 or member.is_dir():
                    continue
                relative = PurePosixPath(*parts[1:])
                if (relative.parts[0] not in {'scripts', 'notebooks', 'data', 'docs',
                                             'requirements-datasets.txt', 'requirements-vision.txt',
                                             'EMPEZAR_AQUI.txt'}
                        or any(part.startswith('.') for part in relative.parts)
                        or relative.suffix.lower() in {'.pt', '.db', '.sqlite3', '.key', '.pem'}):
                    raise ValueError('Archivo ajeno al paquete de Colab: ' + str(relative))
                target = root / str(relative)
                if relative in seen or not target.resolve().is_relative_to(root):
                    raise ValueError('Ruta repetida o externa en el ZIP: ' + str(relative))
                seen.add(relative)
                entries.append((member, target, relative.parts[0] == 'data'))
            if not all(PurePosixPath(name) in seen for name in PROJECT_FILES):
                raise ValueError('El ZIP no contiene el flujo completo. Usa colab_yolo26_listo.zip actualizado.')
            # Check all paths before writing; never follow existing linked directories.
            for _, target, _ in entries:
                if target.is_symlink() or any(p.is_symlink() for p in target.parents if p != root and p.is_relative_to(root)):
                    raise ValueError('Destino enlazado: ' + str(target))
                if target.exists() and not target.is_file():
                    raise ValueError('El destino no es un archivo: ' + str(target))
            preserved = 0
            for member, target, is_data in entries:
                if is_data and target.exists():
                    preserved += 1
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with package.open(member) as source, target.open('wb') as destination:
                    shutil.copyfileobj(source, destination)
        print('Proyecto extraído desde:', archive)
        if preserved:
            print('Datos existentes conservados:', preserved, '| Verifícalos en 00, celda 4.')

    os.environ['IOTTESIS_ROOT'] = str(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    importlib.invalidate_caches()
    print('Proyecto:', root)
    return root

PROJECT_ROOT = iniciar_colab()


## Celda 02 · Instalar dependencias de esta sesión

Instala visión en la sesión actual. Si ya se había importado otra versión de Ultralytics, reinicia el kernel y vuelve a empezar.


In [ ]:
# Celda 02 · Instalar dependencias de esta sesión
%pip install -q -r "{PROJECT_ROOT / 'requirements-vision.txt'}"


## Celda 03 · Comprobar entorno y cargar el flujo

El estado se calcula a partir de datasets y checkpoints guardados; ejecutar una celda no cuenta como entrenamiento completado. En 01–04 selecciona GPU T4. Cambiar el entorno reinicia las variables: repite las celdas 1–3.


In [ ]:
# Celda 03 · Comprobar entorno y cargar el flujo
import json
import torch
import ultralytics
from scripts.entrenar_yolo26 import training_plan
from scripts.flujo_colab import (
    inspect_progress, print_progress, run_or_reuse, latest_experiment,
    freeze_detector, final_records, evaluate_once, comparable_detectors, write_summary,
)
if tuple(int(v) for v in ultralytics.__version__.split('.')[:2]) < (8, 4):
    raise RuntimeError('Instala Ultralytics >=8.4.0 y reinicia el kernel.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No hay una GPU CUDA disponible. En Colab selecciona GPU T4 en el tipo de entorno y vuelve a ejecutar las celdas 1, 2 y 3.')
DEVICE = 0
print('GPU:', torch.cuda.get_device_name(DEVICE), '| Dispositivo CUDA:', DEVICE)

print('Ultralytics:', ultralytics.__version__, '| PyTorch:', torch.__version__, '| Dispositivo:', DEVICE)
print_progress(inspect_progress(PROJECT_ROOT))


## Celda 04 · Recuperar modelos guardados y fijar la evaluación

Evaluación en GPU. Revisa las rutas y las métricas de validación. Si todavía vas a cambiar parámetros o comparar familias, vuelve a 01–03 y no ejecutes test.


In [ ]:
# Celda 04 · Recuperar modelos guardados y fijar la evaluación
if 'PROJECT_ROOT' not in globals():
    raise RuntimeError('El entorno perdió sus variables. Ejecuta las celdas 1, 2 y 3 de este cuaderno y luego repite la 4.')
import json
from scripts.entrenar_yolo26 import training_plan
from scripts.flujo_colab import (
    inspect_progress, print_progress, run_or_reuse, latest_experiment,
    freeze_detector, final_records, evaluate_once, comparable_detectors, write_summary,
)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No hay una GPU CUDA disponible. En Colab selecciona GPU T4 en el tipo de entorno y vuelve a ejecutar las celdas 1, 2 y 3.')
DEVICE = 0
print('GPU:', torch.cuda.get_device_name(DEVICE), '| Dispositivo CUDA:', DEVICE)
FAMILY_CLASIFICADOR = 'yolo26'
FAMILY_DETECTOR = 'yolo26'
USAR_COMPARACION_SI_EXISTE = True
CONFIRMAR_MODELOS_FIJADOS = False  # Cambiar a True tras revisar validación.
records = final_records(PROJECT_ROOT, FAMILY_CLASIFICADOR, FAMILY_DETECTOR,
                        use_comparison=USAR_COMPARACION_SI_EXISTE)
for record in records:
    print(record['task'], record['family'], record['best_weights'])
    print('Validación:', record['validation'])


## Celda 05 · Evaluar test final una vez

Si test ya está registrado, se muestra sin volver a evaluarlo. Si falta y la confirmación anterior sigue en False, la celda se detiene con instrucciones.


In [ ]:
# Celda 05 · Evaluar test final una vez
evaluated = [evaluate_once(record, confirmed=CONFIRMAR_MODELOS_FIJADOS, device=DEVICE)
             for record in records]
for record in evaluated:
    print(record['task'], record['family'], 'TEST:', record['test'])


## Celda 06 · Guardar resumen y mostrar archivos de entrega

Clasificación y detección son tareas diferentes: no comparar accuracy con mAP como si fueran la misma métrica. El registro conserva resultados y versiones.


In [ ]:
# Celda 06 · Guardar resumen y mostrar archivos de entrega
directory = PROJECT_ROOT / 'runs/datasets_reales'
summary = write_summary(evaluated, PROJECT_ROOT)
for record in evaluated:
    best = Path(record['best_weights'])
    print('PESOS:', best)
    print('METADATOS:', best.with_suffix('.metadata.json'))
    print('REGISTRO:', Path(record['save_dir']) / 'experiment.json')
print('Resumen:', summary)
print_progress(inspect_progress(PROJECT_ROOT))


## Celda 07 · Evaluar los demás detectores de 03 — OPCIONAL

Solo después de congelar la selección en 03. No vuelve a elegir el ganador a partir del test ni entrena familias faltantes.


In [ ]:
# Celda 07 · Evaluar los demás detectores de 03 — OPCIONAL
EVALUAR_OTROS_DETECTORES = False
if EVALUAR_OTROS_DETECTORES:
    reference = next(r for r in records if r['task'] == 'detect')
    others = comparable_detectors(PROJECT_ROOT, reference)
    existing = {r['best_weights'] for r in evaluated}
    evaluated += [evaluate_once(r, confirmed=CONFIRMAR_MODELOS_FIJADOS, device=DEVICE)
                  for r in others if r['best_weights'] not in existing]
    print('Resumen ampliado:', write_summary(evaluated, PROJECT_ROOT))
else:
    print('Omitido: solo se evalúan el clasificador y el detector seleccionado.')


## Celda 08 · Medir latencia CPU — OPCIONAL

Usa imágenes de validación, calentamiento y tiempo medido; no inicia entrenamiento.


In [ ]:
# Celda 08 · Medir latencia CPU — OPCIONAL
MEDIR_LATENCIA_CPU = False
if MEDIR_LATENCIA_CPU:
    from scripts.entrenar_yolo26 import benchmark_cpu
    for record in evaluated:
        print(record['task'], benchmark_cpu(record, samples=10))
else:
    print('Omitido: no es necesario para obtener best.pt.')


## Celda 09 · Exportar ONNX — OPCIONAL

Solo si necesitas ONNX. Su salida no sustituye automáticamente el modelo .pt ni configura el backend.


In [ ]:
# Celda 09 · Exportar ONNX — OPCIONAL
EXPORTAR_ONNX = False
if EXPORTAR_ONNX:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnx', 'onnxruntime'], check=True)
    from ultralytics import YOLO
    for record in evaluated:
        output = YOLO(record['best_weights']).export(format='onnx', imgsz=record['imgsz'], device='cpu')
        print('ONNX:', output)
else:
    print('Omitido: el backend actual puede usar directamente los pesos .pt.')


## Qué queda después de este cuaderno
Copia al servidor el checkpoint que vayas a usar y su archivo de metadatos.
Configura `MODEL_WEIGHTS_PATH` con la ruta real y verifica una imagen propia.
Los pilotos foliares no calculan daño por segmentación ni incluyen insectos.
Falta test local de invernadero, revisión agronómica y validación de integración.
No se envían órdenes al ESP32 desde estos cuadernos.
